# Simulation Surrogate Lab • Research workflow
**A thin notebook over a functional Python package**

Train one selected continuous response using 12 continuous factors and repeated simulation runs. This notebook configures and orchestrates the package; the implementation lives in focused, testable modules under `src/surrogate_lab`.

The saved outputs use **synthetic demonstration data**, not your simulation results. Smoke mode exercises nine core model families and all compatible baseline branches. Broad mode adds more trials. A later user-data run requires your factor definitions, bounds, feasibility rule, and response tolerance.

**Workflow:** audit → group-isolated partitions → broad model/feature search → regional validation → error diagnosis → frozen predictor → uncertainty calibration → simulation proposals → independent test evaluation.

Mean predictions and intervals for an individual future run are different outputs. Approximate bootstrap spread is explicitly labeled and is not a guaranteed confidence interval for the mean.

## 1. Install and configure
Extract the complete project archive, then run from the project directory:
```bash
python -m pip install -e ".[notebook,test,parquet]"
python -m pytest
jupyter lab Surrogate_Workflow.ipynb
```
Optional optimizer support: `python -m pip install -e ".[optimization]"`.

No installation or simulation execution is triggered by this notebook. Checkpoint JSON files are written during search; additional result exports are opt-in. Keep the notebook beside the project or install the package in its kernel environment.

In [ ]:
from dataclasses import replace
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display, Markdown

from surrogate_lab import Config, Recipe, config_from_bounds, __version__
from surrogate_lab.data import load_runs, summarize_runs
from surrogate_lab.workflow import prepare_study, develop, box_feasible, nested_evaluate
from surrogate_lab.search import make_recipes, optuna_refine
from surrogate_lab.selection import seed_stability
from surrogate_lab.diagnostics import diagnostic_table, learning_curves, interaction_screen
from surrogate_lab.uncertainty import fit_uncertainty, calibrate_runs, predict_new, final_evaluation
from surrogate_lab.design import propose_batch
from surrogate_lab.storage import export_study, package_fingerprint
from surrogate_lab.plots import plot_diagnostics, plot_learning_curves, plot_interactions, plot_surface, plot_final

# Demonstration input schema only. Replace this mapping with YOUR factors and
# physical bounds; one, three, seventeen, etc. continuous factors are supported.
FACTOR_BOUNDS = {f"factor_{i}": (0., 1.) for i in range(1, 13)}
CFG = config_from_bounds(FACTOR_BOUNDS, profile="smoke", seed=2026, workers=1, bootstrap_members=8)
# Do not configure the number of alternatives or repetitions: data determines them.
DATA_PATH = None  # Replace with a CSV or Parquet path; None uses the labeled synthetic fixture.
# For actual data, edit e.g.:
# CFG = replace(CFG, factors=("factor_1", ...), bounds=((low_1, high_1), ...),
#               response="your_response", alternative="alternative", repetition="repetition",
#               tolerance=YOUR_ACCEPTABLE_ABSOLUTE_ERROR, profile="broad")

def feasible(X):
    """Add physical constraints on raw factor values; return one Boolean per row.
    Change CFG.feasibility_id when changing this rule.
    """
    return box_feasible(X)

plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":.18})
pd.set_option("display.max_columns", 12)
print({"package":__version__,"python":platform.python_version(),"sklearn":sklearn.__version__})

### Optional observed-edge support
Before loading data, set `CFG = replace(CFG, split_strategy="boundary", boundary_budget=12, boundary_max_fraction=.2)`. Selection uses observed minima/maxima only, with deterministic ties and deduplication. No corner or space-filling selection is performed.
After preparation inspect `study["boundary"]`: selected groups and all extremes covered, partition sizes, budget conflicts and uncovered extremes. Physical bounds may extend beyond observed extrema; marginal ranges do not ensure joint support.
Selected groups remain in development CV training folds, including nested folds. Edge-holdout stress tests intentionally remove support. Holdouts sample remaining configurations; independent near-edge alternatives are necessary to validate near-edge predictions. Never use final-test outcomes to choose the strategy.

## 2. Audit and reserve alternatives
IDs are metadata, not predictors. Missing/nonfinite values and failed-run ambiguity must be resolved explicitly. Valid zeros stay. Duplicate factor vectors are grouped together across alternative IDs. Unequal repetition counts and single-run configurations are accepted. Within-configuration variance is unknown for a single run; it is never silently set to zero. Counts are discovered from rows, not entered in configuration.

The default split reserves 20% of configurations for calibration and 20% for final testing. All response-driven discovery uses development data only. Independent repetitions and one coherent scenario/simulator regime are assumed. Shared seeds or dependent batches require additional blocking/covariance treatment.

In [ ]:
raw = load_runs(DATA_PATH, CFG)
study = prepare_study(raw, CFG, feasible)
dev = study["development"]
print("SYNTHETIC DEMONSTRATION" if DATA_PATH is None else "USER DATA")
display(pd.DataFrame({"partition": study["partitions"].keys(),
    "unique_configurations": [len(g) for g in study["partitions"].values()]}))
print("Rows:", len(study["runs"]), "| selected response:", CFG.response)
# Analyze only development outcomes before model selection.
from surrogate_lab.diagnostics import response_noise_report
response_columns = [CFG.response]  # Add other continuous responses here.
dev_keys = dev[[CFG.alternative, CFG.repetition]]
raw_dev = raw.merge(dev_keys, on=[CFG.alternative, CFG.repetition], validate="one_to_one")
display(response_noise_report(raw_dev, response_columns, CFG, feasible))
display(study["inventory"])
display(study["summary"][["group", "reps"]].head(10))

## 3. Inspect planned breadth
The core families are ridge, Elastic Net, kernel ridge, random forest, Extra Trees, histogram boosting, SVR, MLP, and Gaussian process. Each compatible feature/training branch receives an initial trial. Exact GP is capped by configuration count. Optional XGBoost is explicitly enabled with `enable_xgboost=True` after installing it.

Feature branches include raw factors, quadratic terms, additive splines, and training-only selected interaction products. Approved domain features can be configured separately. Weighted-run fitting gives each alternative equal total weight. Affine response scaling preserves the mean target; nonlinear target transformations are not silently introduced.
Optional models: `enable_gam`, `enable_ebm`, `enable_xgboost`, `enable_lightgbm`. Install the `additive` and/or `boosting` extras. Unavailable requested imports are reported and omitted from automatic search; explicit recipes fail visibly. Ridge with spline features is an always-available additive baseline. GP trials include Matérn/RBF and isotropic/ARD kernels with repetition mean variances as observation noise.

In [ ]:
planned = make_recipes(CFG, dev.group.nunique())
breadth = pd.DataFrame([{"family":r.family,"features":r.representation,"training":r.training} for r in planned])
display(breadth.groupby(["family","features","training"]).size().rename("planned_trials").reset_index())
print("Planned initial trials:", len(planned), "| CV fits:", len(planned)*CFG.folds)
from surrogate_lab.search import optional_model_status
display(optional_model_status(CFG))

## 4. Run development search and regional challenges
Random search is branch-balanced. Broad mode adds family-budgeted trials and further branch refinement. Checkpoints resume only matching data, source, configuration, recipes, splits, and environment. Time limits are soft, not hard fit termination.

Finalists are selected using distributed CV RMSE plus `CFG.regional_penalty × worst regional RMSE`. This explicitly declared heuristic can be changed before a study. Regional evaluation includes cluster holdouts and factor-edge holdouts (first two factors in smoke mode; every factor in broad mode). A simple equal-weight ensemble is evaluated using OOF predictions. All displayed development scores remain selection scores.

In [ ]:
development = develop(dev, CFG)
board = development["board"]
finalists = development["finalists"]
selected_recipes = finalists["recipes"]
oof = finalists["oof"]
display(board.head(12).round(3))
display(finalists["ranking"].round(3))
print("Frozen recipes:", [(r.family,r.representation,r.training) for r in selected_recipes])
for phase in ("initial","refinement"):
    result=development[phase]
    if result is not None:
        print(phase, {k:result[k] for k in ("planned","unstarted","seconds")})
failures = [r for r in development["records"] if r["status"]=="failed"]
print("Failed trials:",len(failures),"| successful trials:",len(board))
if failures: display(pd.DataFrame([{"recipe":r["recipe"],"error":r["error"]} for r in failures]).head())
warning_rows=[{"trial_id":r["trial_id"],"warnings":r.get("warnings",[])} for r in development["records"] if r.get("warnings")]
if warning_rows: display(pd.DataFrame(warning_rows).head())

## 5. Diagnose development errors
Residuals are observed repetition mean minus OOF prediction. They contain both surrogate error and uncertainty in the finite-repetition reference mean. Distance is measured in normalized original factor space. Support-band tables use approximately equal-count bins; ties are broken by row order.

In [ ]:
plot_diagnostics(board,oof,CFG)
plt.show()
display(diagnostic_table(oof).round(3))

## 6. Learning curves and seed stability
One fixed development holdout isolates the effects of more distinct alternatives versus more repetitions. These curves are diagnostic, not precise sample-size recommendations. Seed checks rerun the selected recipes; they do not rerun model selection.
For a crossed allocation experiment, use the prespecified quadratic baseline below. Validation references stay fixed; training subsets are nested for each seed. The existing selected-recipe curves reuse selection folds and are descriptive. Neither experiment establishes performance beyond the observed DOE size.
Repetition values are caps, not required balanced counts: a cap of 5 uses 1, 3, and 5 runs from alternatives with 1, 3, and 8 available runs. Actual run counts and costs are reported. All available rows are retained at the largest cap; there is no truncation to the least-replicated alternative.

In [ ]:
curves=learning_curves(dev,selected_recipes,development["splits"][0],CFG)
plot_learning_curves(curves)
plt.show()
stability=seed_stability(selected_recipes,dev,development["splits"],CFG,seeds=(CFG.seed,CFG.seed+1))
display(stability.round(3))
from surrogate_lab.diagnostics import allocation_learning_curves
allocation = allocation_learning_curves(
    dev, (Recipe("ridge", "quadratic", params=(("alpha", 1.),)),),
    development["splits"][:1] if CFG.profile == "smoke" else development["splits"], CFG,
    seeds=(0, 1) if CFG.profile == "smoke" else (0, 1, 2))
display(allocation.groupby(["alternatives", "repetitions", "simulation_runs", "grain"])["rmse"].agg(["mean", "std"]).reset_index())

## 7. Automated interaction screening
The screen fits an additive-spline pilot and ranks pairwise products by their association with residuals. It is cheap and interpretable but can miss threshold or higher-order interactions. It is **not SHAP, a causal estimate, or a significance test**. The raw nonlinear and complete quadratic candidates remain available to avoid relying on this screen alone.

This display uses all development data after selection for interpretation. The selected-feature candidates were fitted separately inside every training fold.

In [ ]:
interactions=interaction_screen(dev,CFG)
display(interactions.head(8).round(3))
plot_interactions(interactions,CFG)
plt.show()

## 8. Fit uncertainty and calibrate future-run intervals
The predictor is frozen before this step. A hierarchical bootstrap refits its fixed recipe(s), sampling configurations and repetitions. The resulting mean spread is approximate and excludes full selection uncertainty. A separate variance model estimates run variability.

Calibration uses exactly one randomly selected run per calibration alternative. Finite-sample conformal quantiles scale prediction intervals for **one future run at a new configuration**. Classical marginal coverage needs exchangeability; fixed/space-filling/adaptive designs and shared random numbers do not automatically satisfy it. Calibration against individual runs does not validate mean confidence intervals. If there are too few calibration units, the correct finite-sample interval can be unbounded.
With partially replicated data, the variance model uses only configurations with measurable within-configuration variance. With no replication, the separate run-noise estimate is unavailable; calibration uses absolute residuals with a constant scale of 1. Future-run intervals remain possible, but noise cannot be separated from model error. GP fitting uses training-only pooled variance for singletons when replication exists, or a learned shared-noise kernel when it does not.

In [ ]:
uncertainty=fit_uncertainty(dev,selected_recipes,CFG)
calibrator=calibrate_runs(uncertainty,study["calibration"],CFG)
print("Calibration units:",len(calibrator["scores"]),"| target:",calibrator["target"])
print("Calibrated multiplier:",round(calibrator["q"],3))
print("Noise estimation:", uncertainty["noise_status"], "| replicated training alternatives:", uncertainty["replicated_alternatives"])
if len(CFG.factors) >= 2 and not interactions.empty:
    pair=tuple(CFG.factors.index(interactions.iloc[0][c]) for c in ("factor_a","factor_b"))
    plot_surface(uncertainty["bundle"],summarize_runs(dev,CFG),pair,CFG,feasible)
    plt.show()
else:
    print("Pairwise surface omitted: this study has one factor.")


The surface above is a **conditional slice** with other factors fixed at development medians. White points are projected design locations; they do not establish support at the full 12-dimensional slice. Smooth-looking contours can interpolate piecewise model predictions and should not be interpreted as proof of smooth physics.

## 9. Propose the next simulation batch
New candidates combine ranked bootstrap spread, distance, and predicted absolute development OOF error. Greedy diversity spreads the batch. Repetition proposals prioritize the estimated local reduction in sample-mean variance. Separate budgets are configured for new alternatives and repetitions; this is a heuristic allocation, not an integrated-error optimum.

Calibration/test configurations are excluded using their **factor values only**. Their outcomes never influence proposals. No simulator runs automatically. A new batch must be audited and incorporated deliberately; reusing integer group IDs after adding configurations is unsafe because their ordering may change.

In [ ]:
reserved=study["summary"].loc[~study["summary"].group.isin(study["partitions"]["development"]),list(CFG.factors)].to_numpy()
proposal=propose_batch(uncertainty,dev,oof,CFG,feasible,reserved)
display(proposal[["action","group","repetitions","acquisition_score","expected_local_mean_variance_reduction","estimated_cost"]].round(3))
print("Proposed runs:",int(proposal.repetitions.sum()),"| no simulator invoked")
# Alternatives with only one observation are prioritized for pilot replication;
# their variance-reduction estimate remains unknown rather than invented.

## 10. Independent final evaluation
This is the first use of final-test response values for performance assessment. If these outcomes guide another revision, they become development evidence and fresh confirmation data are needed.

Mean scores compare with noisy repetition means. Interval coverage uses one held-out run per test alternative, matching the calibration target. The predeclared coverage curve is diagnostic; it does not retune the model. Small samples make regional coverage estimates imprecise.

In [ ]:
final=final_evaluation(uncertainty,calibrator,study["test"],CFG,feasible)
display(pd.DataFrame([final["mean_metrics"]]).round(3))
print("Future-run coverage:",round(final["run_coverage"],3),"| mean interval width:",round(final["run_interval_width"],3))
plot_final(final,calibrator,CFG)
plt.show()
if DATA_PATH is None:
    truth=raw.groupby(list(CFG.factors))["_synthetic_true_mean"].first()
    check=final["means"].merge(truth.rename("true_mean").reset_index(),on=list(CFG.factors))
    print("Synthetic-only latent-mean RMSE:",round(float(np.sqrt(np.mean((check.true_mean-check.predicted_mean)**2))),3))
display(pd.DataFrame([final["run_metrics"]]))
display(final["regional_coverage"])

## 10B. Post-model investigation: predicted versus observed
**Axes: observations on x; predictions on y.** A tight horizontal band means little prediction variation. It is evidence of near-constant predictions, not by itself proof of underfitting.

This read-only phase compares training and held-out results at two grains: individual runs and alternative means. It measures prediction spread, slope, correlation, errors, bias, and improvement over the **development alternative-mean baseline**. It also checks transformed features and estimates how much reference-mean variation may arise from stochastic sampling noise.

Descriptive defaults: SD ratio below 0.10 flags a near-flat band; below 0.50 flags compression; MSE improvement below 5% flags little benefit over the baseline. These are configurable screening heuristics, not significance tests. A correct conditional mean can have less spread than noisy observations. A constant observed response makes correlation, spread ratios and R² undefined; it does not automatically imply model failure.

The equal-alternative noise estimate subtracts average sampling variance from variance among alternative means. It assumes independent repetitions; negative estimates and a small test set indicate uncertainty, not proof of no signal. Individual-run metrics weight rows, while mean metrics weight alternatives equally.

**Interpretation guide:** both partitions flat → check weak signal, preprocessing and alignment; training good but held-out poor → check overfitting and support; run plot broad but mean plot tighter → investigate simulation noise. Warnings propose checks, never establish causes.

Because this examines the final test set, do not use it to tune and then reuse that set as independent evidence. Changes motivated here require fresh confirmation. Key/factor checks cannot detect upstream labels that were already attached to the wrong simulation run.

The additional tables inspect marginal and pairwise DOE coverage, box discrepancy, support distances, residual patterns across factors/predictions/support, and response-tail errors. Tail thresholds come from development means. Empty tails and small regional counts are retained; these are descriptive checks, not acceptance tests. A constrained feasible region may legitimately leave box bins empty.
With single-run alternatives, the overall signal/noise decomposition is marked unavailable. Statistics computed from the replicated subset are labeled accordingly. One factor has no pairwise combinations; small datasets may yield very small test regions or unbounded calibrated intervals.

In [ ]:
from surrogate_lab.postdiagnostics import postfit_report, DiagnosticThresholds
from surrogate_lab.plots import plot_postfit

post_report = postfit_report(
    uncertainty["bundle"], dev, study["test"], CFG,
    thresholds=DiagnosticThresholds(), observed_predictions=final["means"])
display(post_report["scores"].set_index(["partition", "grain"]).T.round(3))
display(post_report["findings"].style.set_properties(**{"white-space": "normal", "text-align": "left"}).hide(axis="index"))
display(post_report["noise"].round(4))
display(post_report["inputs"])
plot_postfit(post_report, CFG)
plt.show()
# Optional standalone diagnostic tables (no automatic retraining):
# post_report["scores"].to_csv("prediction_diagnostic_scores.csv", index=False)
# post_report["findings"].to_csv("prediction_diagnostic_findings.csv", index=False)

display(post_report["coverage"]["summary"])
display(post_report["coverage"]["marginal"])
display(post_report["coverage"]["pairwise"].sort_values("occupied_cell_fraction").head(10))
display(post_report["residual_patterns"])
display(post_report["tails"])

## 11. Predict and export
Supply physical values in the configured factor order. Predictions outside bounds or known feasibility rules are rejected. In-range predictions can still be poorly supported. Bootstrap mean spread and calibrated run intervals retain distinct column names.

Setting `CFG.export_results=True` writes model state, provenance, leaderboard, OOF/test predictions, and batch proposals to `surrogate_results`. Model artifacts are trusted-local joblib files; never load unknown pickle/joblib files.

In [ ]:
query=study["summary"].loc[[0],list(CFG.factors)].to_numpy()
display(predict_new(uncertainty,calibrator,query,CFG,feasible).round(3))
print("Factor order:",CFG.factors)
if CFG.export_results:
    output=export_study("surrogate_results",CFG,study["runs"],study["partitions"],development["records"],board,
                        finalists,uncertainty,calibrator,proposal,final)
    print("Exported:",output)

## 12. Optional deeper evaluation and optimization
These are explicit opt-in calls because they can multiply runtime substantially:
```python
# Full outer evaluation of the selection procedure, using development data only:
# nested = nested_evaluate(dev, CFG, outer_folds=3)

# Optional Optuna TPE for one chosen compatible branch (install optimization extra):
# tuning = optuna_refine(selected_recipes[0], dev, development["splits"], CFG,
#                        package_fingerprint(), trials=30)
```
TPE helper supports ridge, forests, Extra Trees, SVR, and histogram boosting. Other families use the main conditional search. No hard timeout, distributed cluster, SHAP/Sobol, or covariance-aware common-seed implementation is implied. See README and module docstrings for limitations and tests actually run in `VALIDATION.md`.

### References
- [Grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Preprocessing leakage](https://scikit-learn.org/stable/common_pitfalls.html)
- [Optuna optimization](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/003_efficient_optimization_algorithms.html)
- [Stochastic kriging](https://users.iems.northwestern.edu/~nelsonb/Publications/AnkenmanNelsonStaumSK.pdf)
- [Conformal coverage assumptions](https://academic.oup.com/jrsssb/article/87/4/1100/8058684)